# 🚀 KAGGLE STAGE-2 MAX-PING: SEGFORMER-B2 (MiT-B2) — 896×504 + LOVÁSZ-IOU + DICE + EMA (GPU T4 × 2)

### 🔥 5 Nâng cấp cực hạn so với Stage 1 (`mIoU 73.23%`, `Lane IoU 40.21%`) — GIỮ NGUYÊN 100% FPS TRÊN WEB:
1. **Nối tiếp trực tiếp từ `segformer_best.pth` Stage 1 (`73.23% mIoU`)** — không train lại từ đầu.
2. **Nâng độ phân giải `768×432 → 896×504` (+36% điểm ảnh, chuẩn 16:9 chia hết cho 32):** Bắt vạch kẻ mảnh ở xa cực sắc nét mà vẫn giữ nguyên kiến trúc `MiT-B2` (`104.5 MB`, 0 ms tăng thêm trên Web).
3. **Hàm mất mát 3 tầng `Weighted CE [1.0, 1.5, 2.6] + Dice + Lovász-Softmax`:** Hạ trọng số Lane từ `4.0 → 2.6` kết hợp **Lovász-Softmax** (tối ưu trực tiếp chỉ số `IoU`!) để siết gọn viền vạch kẻ, đẩy **`Lane Precision` & `Lane IoU`** nhảy vọt!
4. **Model EMA (Exponential Moving Average) + Cosine LR Decay (`2.5e-5 → 1e-6`) + `DataParallel` 2× T4 FP16:** Chạy nhanh gấp ~7× máy cá nhân, tự kiểm soát mốc `10.2h` an toàn tuyệt đối trước giới hạn 12h Kaggle.
5. **Hỗ trợ TỰ ĐỘNG cả 2 định dạng BDD100K trên Kaggle:** Tự động dùng PNG mask nếu dataset có sẵn, hoặc tự động rasterize trực tiếp từ `poly2d` trong `bdd100k_labels_images_*.json` (cùng bộ **`solesensei/solesensei_bdd100k`** dùng chung với YOLO!).

### ⚡ Hướng dẫn chạy trên Kaggle (chỉ cần đúng 2 Input chuẩn, KHÔNG cần thêm bộ data lạ):
1. **Accelerator:** Chọn **`GPU T4 x2`** | **Internet:** Bật **`ON`**.
2. **Add Input** bên cột phải, gắn đúng 2 nguồn:
   - Dataset chứa file **`segformer_best.pth`** mới nhất (`104.48 MB` — `73.23% mIoU`).
   - **`solesensei/solesensei_bdd100k`** (hoặc bộ BDD100K PNG `drivable` + `lane` của bạn).
3. Bấm **Save Version → Save & Run All (Commit)**. Toàn bộ `segformer_best.pth` + CSV + biểu đồ sẽ xuất gọn tại `/kaggle/working/FINAL_EXPORT/`.

In [ ]:
# ==============================================================================
# BƯỚC 1: KHỞI TẠO MÔI TRƯỜNG & CẤU HÌNH STAGE-2 MAX-PING TRÊN 2× GPU T4
# ==============================================================================
import os, sys, glob, time, math, copy, json, csv, random, shutil
from pathlib import Path
import cv2
cv2.setNumThreads(0)
cv2.ocl.setUseOpenCL(False)

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from transformers import SegformerConfig, SegformerForSemanticSegmentation

CONFIG = {
    'input_root': '/kaggle/input',
    'export_dir': '/kaggle/working/FINAL_EXPORT',
    'pretrained_name': 'nvidia/segformer-b2-finetuned-cityscapes-1024-1024',
    'num_classes': 3,

    # Nâng độ phân giải lên 896x504 (tỷ lệ 16:9 chuẩn, chia hết cho 32)
    'img_w': 896,
    'img_h': 504,
    'lane_dilate': 2,

    # Siêu tham số Stage-2 Polish (hạ LR mượt + siết viền vạch kẻ bằng weight 2.6 + Lovász-IoU)
    'epochs': 4,
    'batch_size': 8 if torch.cuda.device_count() >= 2 else 4,  # 4 ảnh/GPU x 2 T4 = 8 (100% chống OOM ở 896x504)
    'grad_accum_steps': 2,       # Effective batch size = 16
    'lr': 2.5e-5,
    'min_lr': 1e-6,
    'weight_decay': 0.01,
    'warmup_ratio': 0.05,
    'class_weights': [1.0, 1.5, 2.6],
    'ema_decay': 0.999,
    'eval_every_steps': 2000,    # Đánh giá & lưu kỷ lục mỗi ~2,000 bước (16,000 ảnh)
    'time_limit_hours': 10.2,    # Tự dừng êm trước giới hạn 12h Kaggle để chốt eval + xuất file!
    'max_val_samples': 2000,     # Đánh giá chuẩn trên 2,000 ảnh Val (gấp đôi Stage 1 -> độ tin cậy cực cao)
    'num_workers': 4,
    'seed': 42,
}

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CONFIG['seed'])
os.makedirs(CONFIG['export_dir'], exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_GPUS = torch.cuda.device_count()
print(f'✅ Device: {device} | GPUs: {NUM_GPUS} | Batch: {CONFIG["batch_size"]} x accum {CONFIG["grad_accum_steps"]} | Res: {CONFIG["img_w"]}x{CONFIG["img_h"]}')

In [ ]:
# ==============================================================================
# BƯỚC 2: TỰ ĐỘNG QUÉT WEIGHT STAGE 1 + TỰ ĐỘNG NẠP BDD100K (HỖ TRỢ CẢ PNG MASK & JSON POLY2D!)
# ==============================================================================
INPUT_ROOT = CONFIG['input_root']

# 2.1. Tìm file trọng số segformer_best.pth Stage 1
pth_files = sorted(glob.glob(f'{INPUT_ROOT}/**/*.pth', recursive=True))
pref_pth = [p for p in pth_files if 'best' in os.path.basename(p).lower() or 'segformer' in os.path.basename(p).lower()]
STAGE1_CKPT = pref_pth[0] if pref_pth else (pth_files[0] if pth_files else None)
print(f'🎯 Trọng số Stage-1 khởi điểm: {STAGE1_CKPT}')

# 2.2. Quét đệ quy ảnh BDD100K đúng nhánh /100k/ (phân biệt rõ với /10k/)
def build_bdd_img_index(split_name):
    idx_100k, idx_fb = {}, {}
    for root, dirs, files in os.walk(INPUT_ROOT):
        norm = root.replace('\\', '/').lower() + '/'
        if 'label' in norm or 'drivable' in norm or 'lane' in norm or 'mask' in norm:
            continue
        if f'/{split_name}/' not in norm:
            continue
        jpgs = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        if not jpgs:
            continue
        if '/100k/' in norm and '/10k/' not in norm:
            for f in jpgs:
                idx_100k[os.path.splitext(f)[0]] = os.path.join(root, f)
        elif 'bdd' in norm:
            for f in jpgs:
                idx_fb[os.path.splitext(f)[0]] = os.path.join(root, f)
    chosen = idx_100k if len(idx_100k) >= len(idx_fb) else idx_fb
    print(f'🔍 [Images-{split_name.upper()}] Tìm thấy {len(chosen):,} ảnh (100k={len(idx_100k):,}, fallback={len(idx_fb):,})')
    return chosen

def build_png_mask_index(keyword, split_name):
    idx = {}
    for root, dirs, files in os.walk(INPUT_ROOT):
        norm = root.replace('\\', '/').lower() + '/'
        if keyword not in norm or 'color' in norm:
            continue
        if f'/{split_name}/' not in norm and not norm.endswith(f'/{keyword}/'):
            continue
        for f in files:
            if f.lower().endswith('.png'):
                stem = os.path.splitext(f)[0]
                for suf in ('_drivable_id', '_drivable_color', '_drivable', '_lane_id', '_lane_color', '_lane'):
                    stem = stem.replace(suf, '')
                idx[stem] = os.path.join(root, f)
    return idx

class BDD100KStage2Dataset(Dataset):
    """Hỗ trợ tự động 2 chế độ: PNG Masks (nhanh nhất) HOẶC JSON poly2d (dùng chung solesensei_bdd100k!)."""
    def __init__(self, mode, samples, img_w=896, img_h=504, lane_dilate=2, is_train=True):
        self.mode = mode
        self.samples = samples
        self.img_w, self.img_h = img_w, img_h
        self.is_train = is_train
        self.mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        self.std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
        self.dilate_kernel = (
            cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (lane_dilate * 2 + 1, lane_dilate * 2 + 1))
            if lane_dilate > 0 else None
        )

    def __len__(self):
        return len(self.samples)

    def _augment(self, img, mask):
        if random.random() < 0.5:
            img = cv2.flip(img, 1)
            mask = cv2.flip(mask, 1)
        if random.random() < 0.5:
            scale = random.uniform(0.85, 1.15)
            nh, nw = int(self.img_h * scale), int(self.img_w * scale)
            img_s = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
            mask_s = cv2.resize(mask, (nw, nh), interpolation=cv2.INTER_NEAREST)
            if scale >= 1.0:
                y0 = random.randint(0, nh - self.img_h)
                x0 = random.randint(0, nw - self.img_w)
                img = img_s[y0:y0 + self.img_h, x0:x0 + self.img_w]
                mask = mask_s[y0:y0 + self.img_h, x0:x0 + self.img_w]
            else:
                p_img = np.zeros((self.img_h, self.img_w, 3), dtype=img.dtype)
                p_mask = np.zeros((self.img_h, self.img_w), dtype=mask.dtype)
                y0 = random.randint(0, self.img_h - nh)
                x0 = random.randint(0, self.img_w - nw)
                p_img[y0:y0 + nh, x0:x0 + nw] = img_s
                p_mask[y0:y0 + nh, x0:x0 + nw] = mask_s
                img, mask = p_img, p_mask
        if random.random() < 0.6:
            alpha = random.uniform(0.75, 1.25)
            beta = random.uniform(-22, 22)
            img = np.clip(img.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)
        return img, mask

    def _load_mask_png(self, dp, lp):
        drv = cv2.imread(dp, cv2.IMREAD_GRAYSCALE)
        lane = cv2.imread(lp, cv2.IMREAD_GRAYSCALE)
        drv = cv2.resize(drv, (self.img_w, self.img_h), interpolation=cv2.INTER_NEAREST)
        lane = cv2.resize(lane, (self.img_w, self.img_h), interpolation=cv2.INTER_NEAREST)
        drv_bin = ((drv == 0) | (drv == 1)) if int(drv[0, 0]) == 2 else ((drv > 0) & (drv != 255))
        if lane.max() == 255 and (lane == 255).sum() > (lane == 0).sum():
            lane_bin = (lane < 255).astype(np.uint8)
        else:
            lane_bin = (lane > 0).astype(np.uint8)
        if self.dilate_kernel is not None:
            lane_bin = cv2.dilate(lane_bin, self.dilate_kernel, iterations=1)
        mask = np.zeros((self.img_h, self.img_w), dtype=np.uint8)
        mask[drv_bin] = 1
        mask[lane_bin > 0] = 2
        return mask

    def _load_mask_json(self, drv_polys, lane_polys, orig_w, orig_h):
        sx = self.img_w / float(orig_w)
        sy = self.img_h / float(orig_h)
        mask = np.zeros((self.img_h, self.img_w), dtype=np.uint8)
        for pts in drv_polys:
            arr = np.round(np.array(pts, dtype=np.float32) * [sx, sy]).astype(np.int32)
            if len(arr) >= 3:
                cv2.fillPoly(mask, [arr], 1)
        lane_bin = np.zeros((self.img_h, self.img_w), dtype=np.uint8)
        for pts in lane_polys:
            arr = np.round(np.array(pts, dtype=np.float32) * [sx, sy]).astype(np.int32)
            if len(arr) >= 2:
                cv2.polylines(lane_bin, [arr], isClosed=False, color=1, thickness=2)
        if self.dilate_kernel is not None:
            lane_bin = cv2.dilate(lane_bin, self.dilate_kernel, iterations=1)
        mask[lane_bin > 0] = 2
        return mask

    def __getitem__(self, idx):
        if self.mode == 'png':
            ip, dp, lp = self.samples[idx]
            img = cv2.cvtColor(cv2.imread(ip, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (self.img_w, self.img_h), interpolation=cv2.INTER_LINEAR)
            mask = self._load_mask_png(dp, lp)
        else:
            ip, drv_polys, lane_polys = self.samples[idx]
            raw = cv2.cvtColor(cv2.imread(ip, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
            oh, ow = raw.shape[:2]
            img = cv2.resize(raw, (self.img_w, self.img_h), interpolation=cv2.INTER_LINEAR)
            mask = self._load_mask_json(drv_polys, lane_polys, ow, oh)

        if self.is_train:
            img, mask = self._augment(img, mask)
        img_f = (img.astype(np.float32) / 255.0 - self.mean) / self.std
        return torch.from_numpy(img_f.transpose(2, 0, 1)).float(), torch.from_numpy(mask.astype(np.int64))

def prepare_split(split_name, max_samples=None):
    imgs = build_bdd_img_index(split_name)
    drvs = build_png_mask_index('drivable', split_name)
    lanes = build_png_mask_index('lane', split_name)
    common_png = sorted(set(imgs.keys()) & set(drvs.keys()) & set(lanes.keys()))
    if len(common_png) > 500:
        samples = [(imgs[k], drvs[k], lanes[k]) for k in common_png]
        if max_samples and len(samples) > max_samples:
            samples = random.Random(42).sample(samples, max_samples)
        print(f'✅ [PNG-MODE {split_name.upper()}] Nạp {len(samples):,} mẫu (Drivable + Lane PNG)')
        return 'png', samples

    # Fallback thông minh: đọc trực tiếp poly2d từ JSON của solesensei_bdd100k!
    jsons = glob.glob(f'{INPUT_ROOT}/**/*bdd100k_labels_images_{split_name}.json', recursive=True)
    assert jsons, f'❌ Không tìm thấy nhãn PNG (drivable+lane) hoặc JSON {split_name} trong /kaggle/input!'
    print(f'⏳ [JSON-POLY2D-MODE {split_name.upper()}] Đang trích xuất polygon Drivable + Lane từ {jsons[0]}...')
    with open(jsons[0], 'r', encoding='utf-8') as f:
        data = json.load(f)
    samples = []
    for item in data:
        stem = os.path.splitext(item.get('name', ''))[0]
        ip = imgs.get(stem)
        if not ip:
            continue
        drv_p, lane_p = [], []
        for lbl in (item.get('labels') or []):
            cat = str(lbl.get('category', '')).strip().lower()
            polys = lbl.get('poly2d') or []
            if 'drivable' in cat:
                for p in polys:
                    if p.get('vertices'): drv_p.append(p['vertices'])
            elif 'lane' in cat:
                for p in polys:
                    if p.get('vertices'): lane_p.append(p['vertices'])
        if drv_p or lane_p:
            samples.append((ip, drv_p, lane_p))
    if max_samples and len(samples) > max_samples:
        samples = random.Random(42).sample(samples, max_samples)
    print(f'✅ [JSON-POLY2D-MODE {split_name.upper()}] Nạp thành công {len(samples):,} mẫu!')
    return 'json', samples

tr_mode, tr_samples = prepare_split('train', max_samples=None)
va_mode, va_samples = prepare_split('val', max_samples=CONFIG['max_val_samples'])

train_ds = BDD100KStage2Dataset(tr_mode, tr_samples, CONFIG['img_w'], CONFIG['img_h'], CONFIG['lane_dilate'], is_train=True)
val_ds = BDD100KStage2Dataset(va_mode, va_samples, CONFIG['img_w'], CONFIG['img_h'], CONFIG['lane_dilate'], is_train=False)

# Kiểm tra sanity check 1 mẫu đầu tiên
_, sm = val_ds[0]
print(f'🔍 Sanity Check Val[0]: Drivable={(sm==1).float().mean()*100:.2f}% | Lane={(sm==2).float().mean()*100:.2f}% (Phải > 0%)')

train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True,
                          num_workers=CONFIG['num_workers'], pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=CONFIG['batch_size'], shuffle=False,
                        num_workers=CONFIG['num_workers'], pin_memory=True)

In [ ]:
# ==============================================================================
# BƯỚC 3: LOVÁSZ-SOFTMAX + DICE + WEIGHTED CE + EMA + TRAIN STAGE 2 TRÊN 2× GPU T4
# ==============================================================================
def lovasz_grad(gt_sorted):
    p = len(gt_sorted)
    gts = gt_sorted.sum()
    intersection = gts - gt_sorted.float().cumsum(0)
    union = gts + (1 - gt_sorted).float().cumsum(0)
    jaccard = 1.0 - intersection / union.clamp(min=1e-6)
    if p > 1:
        jaccard[1:p] = jaccard[1:p] - jaccard[0:-1]
    return jaccard

def lovasz_softmax_flat(probs, labels, classes='present'):
    if probs.numel() == 0:
        return probs * 0.0
    C = probs.size(1)
    losses = []
    for c in range(C):
        fg = (labels == c).float()
        if classes == 'present' and fg.sum() == 0:
            continue
        errors = (fg - probs[:, c]).abs()
        errors_sorted, perm = torch.sort(errors, dim=0, descending=True)
        fg_sorted = fg[perm]
        losses.append(torch.dot(errors_sorted, lovasz_grad(fg_sorted)))
    return torch.stack(losses).mean() if losses else probs.sum() * 0.0

class MaxPingSegLoss(nn.Module):
    """Kết hợp 3 hàm mất mát mạnh nhất cho phân đoạn đường & vạch mảnh: CE + Dice + Lovász-IoU."""
    def __init__(self, class_weights=(1.0, 1.5, 2.6), w_ce=0.40, w_dice=0.35, w_lovasz=0.25, smooth=1.0):
        super().__init__()
        self.w_ce, self.w_dice, self.w_lovasz, self.smooth = w_ce, w_dice, w_lovasz, smooth
        self.register_buffer('weights', torch.tensor(class_weights, dtype=torch.float32))

    def forward(self, logits, targets):
        if logits.shape[-2:] != targets.shape[-2:]:
            logits = F.interpolate(logits, size=targets.shape[-2:], mode='bilinear', align_corners=False)
        ce = F.cross_entropy(logits, targets, weight=self.weights)
        probs = F.softmax(logits.float(), dim=1)
        targets_oh = F.one_hot(targets, num_classes=probs.shape[1]).permute(0, 3, 1, 2).float()
        inter = torch.sum(probs * targets_oh, (0, 2, 3))
        card = torch.sum(probs + targets_oh, (0, 2, 3))
        dice = 1.0 - ((2.0 * inter + self.smooth) / (card + self.smooth)).mean()

        # Tính Lovász-Softmax ở lưới 1/2 để siêu nhanh trên T4 nhưng vẫn tối ưu trực tiếp IoU
        p_half = F.interpolate(probs, scale_factor=0.5, mode='bilinear', align_corners=False)
        t_half = F.interpolate(targets.unsqueeze(1).float(), scale_factor=0.5, mode='nearest').squeeze(1).long()
        p_flat = p_half.permute(0, 2, 3, 1).contiguous().view(-1, probs.shape[1])
        t_flat = t_half.contiguous().view(-1)
        lov = lovasz_softmax_flat(p_flat, t_flat, classes='present')
        return self.w_ce * ce + self.w_dice * dice + self.w_lovasz * lov

class ModelEMA:
    def __init__(self, model, decay=0.999):
        self.ema = copy.deepcopy(model).eval()
        for p in self.ema.parameters():
            p.requires_grad_(False)
        self.decay = decay

    @torch.no_grad()
    def update(self, model):
        msd = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
        for k, v in self.ema.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(self.decay).add_(msd[k].detach(), alpha=1.0 - self.decay)
            else:
                v.copy_(msd[k])

@torch.inference_mode()
def evaluate_strict(model, loader, criterion, num_classes=3, use_tta=False):
    model.eval()
    conf = torch.zeros((num_classes, num_classes), dtype=torch.int64, device=device)
    tot_loss, nb = 0.0, 0
    for imgs, masks in tqdm(loader, desc='Validating', leave=False):
        imgs, masks = imgs.to(device, non_blocking=True), masks.to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            logits = F.interpolate(model(pixel_values=imgs).logits, size=masks.shape[-2:], mode='bilinear', align_corners=False)
            if use_tta:
                logits_f = F.interpolate(model(pixel_values=torch.flip(imgs, dims=[3])).logits, size=masks.shape[-2:], mode='bilinear', align_corners=False)
                logits = 0.5 * (logits + torch.flip(logits_f, dims=[3]))
            loss = criterion(logits, masks)
        tot_loss += float(loss.item()); nb += 1
        preds = torch.argmax(logits, dim=1)
        k = (masks >= 0) & (masks < num_classes)
        conf += torch.bincount(num_classes * masks[k] + preds[k], minlength=num_classes ** 2).reshape(num_classes, num_classes)
    cm = conf.cpu().numpy().astype(np.float64)
    tp = np.diag(cm)
    fp = cm.sum(axis=0) - tp
    fn = cm.sum(axis=1) - tp
    iou = tp / np.maximum(tp + fp + fn, 1e-6)
    dice = 2 * tp / np.maximum(2 * tp + fp + fn, 1e-6)
    pa = tp.sum() / np.maximum(cm.sum(), 1e-6)
    lane_prec = tp[2] / max(tp[2] + fp[2], 1e-6)
    lane_rec = tp[2] / max(tp[2] + fn[2], 1e-6)
    return {
        'loss': tot_loss / max(nb, 1), 'pa': float(pa), 'miou': float(iou.mean()), 'dice': float(dice.mean()),
        'iou_bg': float(iou[0]), 'iou_drv': float(iou[1]), 'iou_lane': float(iou[2]),
        'dice_drv': float(dice[1]), 'dice_lane': float(dice[2]),
        'lane_prec': float(lane_prec), 'lane_rec': float(lane_rec),
    }

# Khởi tạo SegFormer-B2 (3 lớp) & nạp trọng số Stage 1
id2label = {0: 'background', 1: 'drivable_area', 2: 'lane_marking'}
label2id = {v: k for k, v in id2label.items()}
base_model = SegformerForSemanticSegmentation.from_pretrained(
    CONFIG['pretrained_name'], num_labels=3, id2label=id2label, label2id=label2id, ignore_mismatched_sizes=True
)
if STAGE1_CKPT and os.path.exists(STAGE1_CKPT):
    ckpt = torch.load(STAGE1_CKPT, map_location='cpu', weights_only=False)
    sd = ckpt.get('model_state_dict', ckpt.get('state_dict', ckpt)) if isinstance(ckpt, dict) else ckpt
    clean_sd = {k.replace('module.', ''): v for k, v in sd.items()}
    missing, unexpected = base_model.load_state_dict(clean_sd, strict=False)
    print(f'✅ Đã nạp trọn vẹn trọng số Stage 1! (missing={len(missing)}, unexpected={len(unexpected)})')

base_model.to(device)
ema = ModelEMA(base_model, decay=CONFIG['ema_decay'])
model = nn.DataParallel(base_model) if NUM_GPUS >= 2 else base_model

criterion = MaxPingSegLoss(class_weights=tuple(CONFIG['class_weights'])).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
total_opt_steps = max(1, (len(train_loader) * CONFIG['epochs']) // CONFIG['grad_accum_steps'])
warmup_steps = int(total_opt_steps * CONFIG['warmup_ratio'])

def lr_lambda(step):
    if step < warmup_steps:
        return float(step + 1) / float(max(1, warmup_steps))
    prog = float(step - warmup_steps) / float(max(1, total_opt_steps - warmup_steps))
    cos_v = 0.5 * (1.0 + math.cos(math.pi * min(1.0, prog)))
    return (CONFIG['min_lr'] / CONFIG['lr']) + (1.0 - CONFIG['min_lr'] / CONFIG['lr']) * cos_v

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

# Đánh giá mốc Baseline (Epoch 0 - ngay trước khi train Stage 2)
t_start = time.time()
base_m = evaluate_strict(ema.ema, val_loader, criterion)
best_miou = base_m['miou']
best_metrics = base_m
print(f"📌 Baseline Stage-1 ở 896x504: PA={base_m['pa']*100:.2f}% | mIoU={base_m['miou']*100:.2f}% | DrvIoU={base_m['iou_drv']*100:.2f}% | LaneIoU={base_m['iou_lane']*100:.2f}% (P={base_m['lane_prec']*100:.2f}%, R={base_m['lane_rec']*100:.2f}%)")

csv_path = os.path.join(CONFIG['export_dir'], 'training_metrics_stage2.csv')
csv_fields = ['step', 'epoch', 'loss', 'pa', 'miou', 'dice', 'iou_bg', 'iou_drv', 'iou_lane', 'dice_drv', 'dice_lane', 'lane_prec', 'lane_rec', 'elapsed_min']
history = []
with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    csv.DictWriter(f, fieldnames=csv_fields).writeheader()

def save_best(ep, metrics, raw_model):
    out_pth = os.path.join(CONFIG['export_dir'], 'segformer_best.pth')
    torch.save({
        'epoch': ep,
        'val_miou': metrics['miou'],
        'metrics': metrics,
        'config': CONFIG,
        'model_state_dict': raw_model.state_dict(),
    }, out_pth)
    raw_model.config.save_pretrained(CONFIG['export_dir'])

save_best(0, base_m, ema.ema)
global_step = 0
stop_early = False

for ep in range(1, CONFIG['epochs'] + 1):
    if stop_early:
        break
    model.train()
    optimizer.zero_grad(set_to_none=True)
    pbar = tqdm(enumerate(train_loader, 1), total=len(train_loader), desc=f'Stage-2 Ep {ep}/{CONFIG["epochs"]}')
    for bi, (imgs, masks) in pbar:
        imgs, masks = imgs.to(device, non_blocking=True), masks.to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            logits = model(pixel_values=imgs).logits
            loss = criterion(logits, masks) / CONFIG['grad_accum_steps']
        scaler.scale(loss).backward()
        if bi % CONFIG['grad_accum_steps'] == 0 or bi == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            ema.update(model)
            global_step += 1

        elapsed_h = (time.time() - t_start) / 3600.0
        if bi % 50 == 0:
            pbar.set_postfix(loss=f'{loss.item()*CONFIG["grad_accum_steps"]:.4f}', lr=f'{scheduler.get_last_lr()[0]:.2e}', hrs=f'{elapsed_h:.2f}h')

        if (bi % CONFIG['eval_every_steps'] == 0) or (bi == len(train_loader)) or (elapsed_h >= CONFIG['time_limit_hours']):
            m = evaluate_strict(ema.ema, val_loader, criterion)
            row = {'step': global_step, 'epoch': ep, 'elapsed_min': round((time.time() - t_start) / 60.0, 2)}
            for k in csv_fields:
                if k in m:
                    row[k] = round(m[k], 4)
            history.append(row)
            with open(csv_path, 'a', newline='', encoding='utf-8') as f:
                csv.DictWriter(f, fieldnames=csv_fields).writerow(row)
            print(f"\n[Ep {ep} | Step {global_step}] Loss={m['loss']:.4f} | PA={m['pa']*100:.2f}% | mIoU={m['miou']*100:.2f}% | Dice={m['dice']*100:.2f}% | Drv={m['iou_drv']*100:.2f}% | Lane={m['iou_lane']*100:.2f}% (P={m['lane_prec']*100:.2f}%, R={m['lane_rec']*100:.2f}%)")
            if m['miou'] > best_miou:
                best_miou = m['miou']
                best_metrics = m
                save_best(ep, m, ema.ema)
                print(f'   🏆 LƯU KỶ LỤC MỚI segformer_best.pth (mIoU={best_miou*100:.2f}%)')
            model.train()
            if elapsed_h >= CONFIG['time_limit_hours']:
                print(f'⏱️ Đạt mốc an toàn {CONFIG["time_limit_hours"]}h — tự dừng êm để xuất FINAL_EXPORT!')
                stop_early = True
                break

In [ ]:
# ==============================================================================
# BƯỚC 4: ĐÁNH GIÁ CHỐT KỶ LỤC (STANDARD & TTA) + VẼ BIỂU ĐỒ + XUẤT FINAL_EXPORT
# ==============================================================================
best_pth = os.path.join(CONFIG['export_dir'], 'segformer_best.pth')
best_ckpt = torch.load(best_pth, map_location=device, weights_only=False)
ema.ema.load_state_dict(best_ckpt['model_state_dict'])

m_std = evaluate_strict(ema.ema, val_loader, criterion, use_tta=False)
m_tta = evaluate_strict(ema.ema, val_loader, criterion, use_tta=True)

summary_txt = (
    '==================================================================\n'
    '📊 SEGFORMER-B2 STAGE-2 MAX-PING (896x504, Lovasz+Dice+CE+EMA):\n'
    f'  • Pixel Accuracy  : {m_std["pa"]*100:.2f}%\n'
    f'  • mIoU (3-class)  : {m_std["miou"]*100:.2f}%\n'
    f'  • Mean Dice       : {m_std["dice"]*100:.2f}%\n'
    f'  • Drivable IoU    : {m_std["iou_drv"]*100:.2f}% (Dice: {m_std["dice_drv"]*100:.2f}%)\n'
    f'  • Lane IoU        : {m_std["iou_lane"]*100:.2f}% (Dice: {m_std["dice_lane"]*100:.2f}%)\n'
    f'  • Lane Prec / Rec : {m_std["lane_prec"]*100:.2f}% / {m_std["lane_rec"]*100:.2f}%\n'
    '------------------------------------------------------------------\n'
    '🏆 SEGFORMER-B2 STAGE-2 + FLIP-TTA (Chấm điểm cực hạn):\n'
    f'  • Pixel Accuracy  : {m_tta["pa"]*100:.2f}%\n'
    f'  • mIoU (3-class)  : {m_tta["miou"]*100:.2f}%\n'
    f'  • Mean Dice       : {m_tta["dice"]*100:.2f}%\n'
    f'  • Drivable IoU    : {m_tta["iou_drv"]*100:.2f}% (Dice: {m_tta["dice_drv"]*100:.2f}%)\n'
    f'  • Lane IoU        : {m_tta["iou_lane"]*100:.2f}% (Dice: {m_tta["dice_lane"]*100:.2f}%)\n'
    f'  • File size       : {os.path.getsize(best_pth)/(1024**2):.2f} MB ({best_pth})\n'
    '==================================================================\n'
)
print(summary_txt)
with open(os.path.join(CONFIG['export_dir'], 'FINAL_METRICS_SUMMARY.txt'), 'w', encoding='utf-8') as f:
    f.write(summary_txt)

if history:
    xs = [r['step'] for r in history]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    axes[0].plot(xs, [r['loss'] for r in history], 'o-', color='#e11d48'); axes[0].set_title('Val Loss'); axes[0].grid(True, alpha=0.3)
    axes[1].plot(xs, [r['miou']*100 for r in history], 'o-', label='mIoU', color='#2563eb')
    axes[1].plot(xs, [r['dice']*100 for r in history], 's-', label='Dice', color='#10b981')
    axes[1].plot(xs, [r['pa']*100 for r in history], '^-', label='PA', color='#f59e0b')
    axes[1].set_title('Overall Metrics (%)'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
    axes[2].plot(xs, [r['iou_drv']*100 for r in history], 'o-', label='Drivable IoU', color='#0284c7')
    axes[2].plot(xs, [r['iou_lane']*100 for r in history], 's-', label='Lane IoU', color='#7c3aed')
    axes[2].plot(xs, [r['lane_prec']*100 for r in history], '^-', label='Lane Prec', color='#16a34a')
    axes[2].set_title('Per-Class IoU & Lane Precision (%)'); axes[2].legend(); axes[2].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG['export_dir'], 'training_curves.png'), dpi=150)
    print('✅ Đã xuất đầy đủ segformer_best.pth + CSV + training_curves.png tại:', CONFIG['export_dir'])